# 🧬 Aura-1B — Entraînement des 4 adaptateurs LoRA

**Runtime → Modifier le type d'exécution → T4 GPU** puis Exécuter tout.

Ce notebook entraîne 4 adaptateurs LoRA (r=16, ~1% des paramètres) sur Llama-3.2-1B-Instruct :

| Adaptateur | Dataset (déjà dans le repo) | Rôle |
|---|---|---|
| `aura-web` | `datasets/web.jsonl` (153 ex.) | journaliste factuel, répond à partir des faits |
| `aura-math` | `datasets/math.jsonl` (123 ex.) | calcul + raisonnement numérique |
| `aura-code` | `datasets/code.jsonl` (78 ex.) | expert Python commenté |
| `aura-general` | `datasets/general.jsonl` (73 ex.) | assistant clair et précis |

Durée totale : ~15-20 min. À la fin : 4 fichiers `.gguf` à déposer dans `adaptateurs/` sur ton PC.

In [ ]:
# 1. Dépendances (~1 min)
# torchao : Colab embarque une version trop vieille pour peft (>0.16 requis)
# et le LoRA ne l'utilise PAS -> on le retire, peft le saute proprement.
!pip -q uninstall -y torchao
!pip -q install peft transformers accelerate bitsandbytes gguf
import torch
print('GPU :', torch.cuda.get_device_name(0))

In [ ]:
# 2. Récupérer les datasets + le script d'entraînement depuis le repo
#    (ré-exécutable : met à jour le clone s'il existe déjà)
!git clone --depth 1 https://github.com/Simonc44/aura-1b.git 2>/dev/null || git -C aura-1b pull --ff-only
%cd aura-1b
!wc -l datasets/*.jsonl

## 🏋️ Entraînement des 4 adaptateurs

In [ ]:
# 3. Boucle d'entraînement : 4 catégories, ~4 min chacune sur T4
import sys
sys.path.insert(0, 'scripts')
sys.path.insert(0, '.')          # aura_personnalites.py est a la racine
from entrainer_adaptateur import entrainer

resultats = {}
for cat in ['web', 'math', 'code', 'general']:
    print(f'\n{"="*60}\n[CAT] {cat}\n{"="*60}')
    resultats[cat] = entrainer(cat, f'datasets/{cat}.jsonl', epochs=3)
print('\n[OK] Entrainements termines :', resultats)

In [ ]:
# 4. Conversion PEFT -> GGUF LoRA (format lu par llama.cpp)
#    Base = miroir public unsloth (poids identiques a meta-llama, sans token)
!git clone --depth 1 --filter=blob:none --sparse https://github.com/ggml-org/llama.cpp.git tmp-llama-cpp
%cd tmp-llama-cpp
!git sparse-checkout set conversion gguf-py --skip-checks
%cd ..

import subprocess, os
env = dict(os.environ, PYTHONPATH='tmp-llama-cpp/gguf-py')
for cat, dossier in resultats.items():
    final = f'tmp-lora-{cat}/final'
    cmd = ['python', 'tmp-llama-cpp/convert_lora_to_gguf.py',
           '--outfile', f'adaptateurs/aura-{cat}.gguf',
           '--outtype', 'q8_0',
           '--base-model-id', 'unsloth/Llama-3.2-1B-Instruct',
           final]
    r = subprocess.run(cmd, env=env, capture_output=True, text=True)
    print(f'[{cat}]', 'OK' if r.returncode == 0 else r.stderr[-500:])
!ls -lh adaptateurs/*.gguf

## 📥 Récupérer les adaptateurs

Le zip contient les 4 fichiers `.gguf`. Extrais-le et dépose les 4 fichiers dans le dossier `adaptateurs/` du projet sur ton PC :

```
C:/Users/admin/Desktop/Dev/IA/aura-1b/adaptateurs/
├── aura-web.gguf
├── aura-math.gguf
├── aura-code.gguf
└── aura-general.gguf
```

Puis côté PC : `AURA_ADAPTATEURS=1` (ex. `$env:AURA_ADAPTATEURS="1"` dans PowerShell avant `python -m aura`).

In [ ]:
# 5. Zip final + telechargement
!zip -j adaptateurs_aura.zip adaptateurs/*.gguf
from google.colab import files
files.download('adaptateurs_aura.zip')